# Notebook 2: Tokenization & Embeddings

## Learning Objectives
- Understand tokenization (BPE, WordPiece)
- Implement token embeddings
- Implement Rotary Position Embedding (RoPE)
- Understand positional encoding

## Task 1: Tokenization Basics

### HINT: Why Tokenize?
Neural networks work with numbers, not text. Tokenization converts text into integer IDs.

Common approaches:
- **BPE** (Byte-Pair Encoding): Merges frequent byte pairs
- **WordPiece**: Similar to BPE, used by BERT
- **SentencePiece**: Unigram model, used by many LLMs

In [ ]:
# Exercise: Simple BPE-style tokenization (simplified)
class SimpleTokenizer:
    """Simplified tokenizer for demonstration"""
    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size
        # In practice, you'd load a real tokenizer
        # from transformers import AutoTokenizer
        
    def encode(self, text):
        # Simplified: just convert chars to IDs
        # Real implementation uses vocabulary lookup
        return [ord(c) % self.vocab_size for c in text]
    
    def decode(self, ids):
        return ''.join(chr(i % 128) for i in ids)

tokenizer = SimpleTokenizer(vocab_size=50000)
text = "Hello DeepSeek!"
tokens = tokenizer.encode(text)
decoded = tokenizer.decode(tokens)
print(f"Original: {text}")
print(f"Tokens: {tokens}")
print(f"Decoded: {decoded}")

## Task 2: Token Embeddings

### HINT: Embedding Layer
Token embeddings map token IDs to dense vectors:
```
input_ids: [1, 2, 3, 4, 5]
→ embeddings → [batch, seq_len, hidden_dim]
```

In [ ]:
import torch
import torch.nn as nn
import math

class TokenEmbedding(nn.Module):
    """Token embedding layer"""
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        
    def forward(self, input_ids):
        return self.embedding(input_ids)

# Test
vocab_size = 50000
hidden_size = 512
seq_len = 8
batch_size = 2

embedding_layer = TokenEmbedding(vocab_size, hidden_size)
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
output = embedding_layer(input_ids)

print(f"Input shape: {input_ids.shape}")
print(f"Output shape: {output.shape}")  # [batch, seq, hidden]

## Task 3: Rotary Position Embedding (RoPE)

### HINT: RoPE Concept
RoPE encodes position information by rotating query/key vectors:

```
RoPE(x_m, m) = x_m * cos(mθ) + rotate(x_m) * sin(mθ)
```

This provides:
- Relative position encoding
- Decays attention with distance
- Works well for variable-length sequences

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """RoPE implementation - rotary position embedding"""
    def __init__(self, dim, max_seq_len=512, base=10000):
        super().__init__()
        self.dim = dim
        self.base = base
        
        # Precompute inverse frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        
    def forward(self, seq_len, device):
        # Create position indices
        positions = torch.arange(seq_len, device=device).float()
        
        # Compute angles: position * inverse_freq
        angles = positions.unsqueeze(1) * self.inv_freq.unsqueeze(0)
        
        # Concatenate cos and sin
        cos = angles.cos()
        sin = angles.sin()
        
        return cos, sin

# Test RoPE
rope = RotaryPositionalEmbedding(dim=64, max_seq_len=512)
cos, sin = rope(seq_len=10, device='cpu')
print(f"Cos shape: {cos.shape}")  # [seq, dim/2]
print(f"Sin shape: {sin.shape}")

## Task 4: Apply RoPE to Q and K

### HINT: RoPE Application
```python
def apply_rope(x, cos, sin):
    # x: [batch, heads, seq, head_dim]
    x1, x2 = x[..., :head_dim//2], x[..., head_dim//2:]
    return torch.cat([
        x1 * cos + x2 * sin,
        x2 * cos - x1 * sin
    ], dim=-1)
```

In [ ]:
def apply_rope(x, cos, sin):
    """Apply rotary position embedding to query/key"""
    # x: [batch, heads, seq, head_dim]
    head_dim = x.size(-1)
    assert head_dim % 2 == 0, "head_dim must be even"
    
    x1 = x[..., :head_dim//2]
    x2 = x[..., head_dim//2:]
    
    # Reshape cos/sin for broadcasting: [seq, head_dim//2] -> [1, 1, seq, head_dim//2]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    
    return torch.cat([
        x1 * cos + x2 * sin,
        x2 * cos - x1 * sin
    ], dim=-1)

# Test with sample Q, K
batch, heads, seq, head_dim = 2, 8, 16, 64
q = torch.randn(batch, heads, seq, head_dim)
k = torch.randn(batch, heads, seq, head_dim)

cos, sin = rope(seq, 'cpu')
q_rope = apply_rope(q, cos, sin)
k_rope = apply_rope(k, cos, sin)

print(f"Q shape after RoPE: {q_rope.shape}")
print(f"K shape after RoPE: {k_rope.shape}")

## Task 5: Complete Embedding Module

### HINT: Putting it together
Combine token embeddings + RoPE for complete embedding layer

In [ ]:
class DeepSeekEmbeddings(nn.Module):
    """Complete embedding module with token + RoPE"""
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(config['vocab_size'], config['hidden_size'])
        self.rope = RotaryPositionalEmbedding(
            config['hidden_size'] // config['num_heads'],
            config['max_position_embeddings']
        )
        
    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        
        # Token embeddings
        hidden_states = self.token_embedding(input_ids)
        
        # RoPE will be applied inside attention
        # (we return embeddings and separately apply RoPE in attention)
        return hidden_states

config = {
    'vocab_size': 50000,
    'hidden_size': 512,
    'num_heads': 8,
    'max_position_embeddings': 512
}

embeddings = DeepSeekEmbeddings(config)
input_ids = torch.randint(0, config['vocab_size'], (2, 8))
output = embeddings(input_ids)
print(f"Input: {input_ids.shape}")
print(f"Output: {output.shape}")

## Verification

Complete these checks:
1. ✅ Can explain why tokenization is needed
2. ✅ Understand token embedding layer
3. ✅ Can implement RoPE from scratch
4. ✅ Can apply RoPE to Q/K vectors